# Skymapper lightcurve import

What follows is a sketch of what steps were taken to perform this import. It occurred over the course of months, and some lessons were learned partway through, and have been (mostly) retro-actively updated.

- When running a job on arnor, prefer `"/astro/store/shiren/hats"` for temp storage
- `byte_pixel_threshold` has had a number of speed improvements since the start of execution, and is only applied at the very last stage.
- the skymaps are moved around post-facto in shell

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import lsdb
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dask.distributed import Client
from hats_import import CollectionArguments, VerificationArguments, pipeline, pipeline_with_client
from hats_import.catalog import ImportArguments
from hats_import.catalog.file_readers import CsvReader, InputReader
from hats_import.margin_cache.margin_cache_arguments import MarginCacheArguments
from pyarrow import csv

lsdb.show_versions()


## Lookups

Schema and visit table.

The photometry table only contains an "image ID" and not the time of the observation, so creating lightcurves kinda sucks.

The (much smaller) "images" table contains all of this data. so let's create a side input from it.

According to: https://skymapper.anu.edu.au/table-browser/

> Table of exposures and telescope information \
> (417,223 rows, 18 columns)

In [ ]:
schema_from_pyarrow = pa.schema(
    [
        pa.field("object_id", pa.int64()),
        pa.field("raj2000", pa.float64()),
        pa.field("dej2000", pa.float64()),
        pa.field("e_raj2000", pa.int64(), nullable=True),
        pa.field("e_dej2000", pa.int64(), nullable=True),
        pa.field("smss_j", pa.string(), nullable=True),
        pa.field("mean_epoch", pa.float64(), nullable=True),
        pa.field("rms_epoch", pa.float64(), nullable=True),
        pa.field("glon", pa.float64(), nullable=True),
        pa.field("glat", pa.float64(), nullable=True),
        pa.field("flags", pa.int32(), nullable=True),
        pa.field("nimaflags", pa.int64(), nullable=True),
        pa.field("ngood", pa.int32(), nullable=True),
        pa.field("u_flags", pa.int32(), nullable=True),
        pa.field("u_nimaflags", pa.int64(), nullable=True),
        pa.field("u_ngood", pa.int32(), nullable=True),
        pa.field("u_nclip", pa.int32(), nullable=True),
        pa.field("v_flags", pa.int32(), nullable=True),
        pa.field("v_nimaflags", pa.int64(), nullable=True),
        pa.field("v_ngood", pa.int32(), nullable=True),
        pa.field("v_nclip", pa.int32(), nullable=True),
        pa.field("g_flags", pa.int32(), nullable=True),
        pa.field("g_nimaflags", pa.int64(), nullable=True),
        pa.field("g_ngood", pa.int32(), nullable=True),
        pa.field("g_nclip", pa.int32(), nullable=True),
        pa.field("r_flags", pa.int32(), nullable=True),
        pa.field("r_nimaflags", pa.int64(), nullable=True),
        pa.field("r_ngood", pa.int32(), nullable=True),
        pa.field("r_nclip", pa.int32(), nullable=True),
        pa.field("i_flags", pa.int32(), nullable=True),
        pa.field("i_nimaflags", pa.int64(), nullable=True),
        pa.field("i_ngood", pa.int32(), nullable=True),
        pa.field("i_nclip", pa.int32(), nullable=True),
        pa.field("z_flags", pa.int32(), nullable=True),
        pa.field("z_nimaflags", pa.int64(), nullable=True),
        pa.field("z_ngood", pa.int32(), nullable=True),
        pa.field("z_nclip", pa.int32(), nullable=True),
        pa.field("class_star", pa.float64(), nullable=True),
        pa.field("chi2_psf", pa.float64(), nullable=True),
        pa.field("flags_psf", pa.int64(), nullable=True),
        pa.field("radius_petro", pa.float64(), nullable=True),
        pa.field("mean_fwhm", pa.float64(), nullable=True),
        pa.field("u_psf", pa.float64(), nullable=True),
        pa.field("e_u_psf", pa.float64(), nullable=True),
        pa.field("u_petro", pa.float64(), nullable=True),
        pa.field("e_u_petro", pa.float64(), nullable=True),
        pa.field("u_apc05", pa.float64(), nullable=True),
        pa.field("e_u_apc05", pa.float64(), nullable=True),
        pa.field("u_mmvar", pa.float64(), nullable=True),
        pa.field("v_psf", pa.float64(), nullable=True),
        pa.field("e_v_psf", pa.float64(), nullable=True),
        pa.field("v_petro", pa.float64(), nullable=True),
        pa.field("e_v_petro", pa.float64(), nullable=True),
        pa.field("v_apc05", pa.float64(), nullable=True),
        pa.field("e_v_apc05", pa.float64(), nullable=True),
        pa.field("v_mmvar", pa.float64(), nullable=True),
        pa.field("g_psf", pa.float64(), nullable=True),
        pa.field("e_g_psf", pa.float64(), nullable=True),
        pa.field("g_petro", pa.float64(), nullable=True),
        pa.field("e_g_petro", pa.float64(), nullable=True),
        pa.field("g_apc05", pa.float64(), nullable=True),
        pa.field("e_g_apc05", pa.float64(), nullable=True),
        pa.field("g_mmvar", pa.float64(), nullable=True),
        pa.field("r_psf", pa.float64(), nullable=True),
        pa.field("e_r_psf", pa.float64(), nullable=True),
        pa.field("r_petro", pa.float64(), nullable=True),
        pa.field("e_r_petro", pa.float64(), nullable=True),
        pa.field("r_apc05", pa.float64(), nullable=True),
        pa.field("e_r_apc05", pa.float64(), nullable=True),
        pa.field("r_mmvar", pa.float64(), nullable=True),
        pa.field("i_psf", pa.float64(), nullable=True),
        pa.field("e_i_psf", pa.float64(), nullable=True),
        pa.field("i_petro", pa.float64(), nullable=True),
        pa.field("e_i_petro", pa.float64(), nullable=True),
        pa.field("i_apc05", pa.float64(), nullable=True),
        pa.field("e_i_apc05", pa.float64(), nullable=True),
        pa.field("i_mmvar", pa.float64(), nullable=True),
        pa.field("z_psf", pa.float64(), nullable=True),
        pa.field("e_z_psf", pa.float64(), nullable=True),
        pa.field("z_petro", pa.float64(), nullable=True),
        pa.field("e_z_petro", pa.float64(), nullable=True),
        pa.field("z_apc05", pa.float64(), nullable=True),
        pa.field("e_z_apc05", pa.float64(), nullable=True),
        pa.field("z_mmvar", pa.float64(), nullable=True),
        pa.field("self_id1", pa.int64(), nullable=True),
        pa.field("self_dist1", pa.float64(), nullable=True),
        pa.field("self_id2", pa.int64(), nullable=True),
        pa.field("self_dist2", pa.float64(), nullable=True),
        pa.field("self_id3", pa.int64(), nullable=True),
        pa.field("self_dist3", pa.float64(), nullable=True),
        pa.field("cnt_self_15", pa.float64(), nullable=True),
        pa.field("ebmv_sfd", pa.float64(), nullable=True),
        pa.field("ebmv_gnilc", pa.float64(), nullable=True),
        pa.field("ebmv_g_err", pa.float64(), nullable=True),
        pa.field("gaia_dr3_id1", pa.int64(), nullable=True),
        pa.field("gaia_dr3_dist1", pa.float64(), nullable=True),
        pa.field("gaia_dr3_id2", pa.int64(), nullable=True),
        pa.field("gaia_dr3_dist2", pa.float64(), nullable=True),
        pa.field("cnt_gaia_dr3_15", pa.int64(), nullable=True),
        pa.field("twomass_key", pa.int64(), nullable=True),
        pa.field("twomass_dist", pa.float64(), nullable=True),
        pa.field("allwise_cntr", pa.int64(), nullable=True),
        pa.field("allwise_dist", pa.float64(), nullable=True),
        pa.field("catwise_id", pa.string(), nullable=True),
        pa.field("catwise_dist", pa.float64(), nullable=True),
        pa.field("refcat2_id", pa.int64(), nullable=True),
        pa.field("refcat2_dist", pa.float64(), nullable=True),
        pa.field("ps1_dr1_id", pa.int64(), nullable=True),
        pa.field("ps1_dr1_dist", pa.float64(), nullable=True),
        pa.field("galex_guv_id", pa.int64(), nullable=True),
        pa.field("galex_guv_dist", pa.float64(), nullable=True),
        pa.field("vhs_dr6_id", pa.int64(), nullable=True),
        pa.field("vhs_dr6_dist", pa.float64(), nullable=True),
        pa.field("ls_dr9_id", pa.int64(), nullable=True),
        pa.field("ls_dr9_dist", pa.float64(), nullable=True),
        pa.field("des_dr2_id", pa.int64(), nullable=True),
        pa.field("des_dr2_dist", pa.float64(), nullable=True),
        pa.field("nsc_dr2_id", pa.string(), nullable=True),
        pa.field("nsc_dr2_dist", pa.float64(), nullable=True),
        pa.field("splus_dr3_id", pa.string(), nullable=True),
        pa.field("splus_dr3_dist", pa.float64(), nullable=True),
    ]
)
schema_file = os.path.join(
    "/epyc/data3/hats/raw/skymapper/", "schema_from_pyarrow.parquet"
)
pq.write_metadata(schema_from_pyarrow, schema_file)

In [ ]:
schema_from_pyarrow = pa.schema(
    [
        pa.field("object_id", pa.int64()),
        pa.field("image_id", pa.int64()),
        pa.field("date", pa.float64()),
        pa.field("fwhm_mean", pa.float64(), nullable=True),
        pa.field("airmass", pa.float64(), nullable=True),
        pa.field("ccd", pa.int32()),
        pa.field("ra_img", pa.float64()),
        pa.field("decl_img", pa.float64()),
        pa.field("x_img", pa.float64(), nullable=True),
        pa.field("y_img", pa.float64(), nullable=True),
        pa.field("flags", pa.int32(), nullable=True),
        pa.field("nimaflags", pa.int64(), nullable=True),
        pa.field("background", pa.float64(), nullable=True),
        pa.field("flux_max", pa.float64(), nullable=True),
        pa.field("mu_max", pa.float64(), nullable=True),
        pa.field("class_star", pa.float64(), nullable=True),
        pa.field("a", pa.float64(), nullable=True),
        pa.field("e_a", pa.float64(), nullable=True),
        pa.field("b", pa.float64(), nullable=True),
        pa.field("e_b", pa.float64(), nullable=True),
        pa.field("pa", pa.float64(), nullable=True),
        pa.field("e_pa", pa.float64(), nullable=True),
        pa.field("elong", pa.float64(), nullable=True),
        pa.field("fwhm", pa.float64(), nullable=True),
        pa.field("radius_petro", pa.float64(), nullable=True),
        pa.field("radius_kron", pa.float64(), nullable=True),
        pa.field("radius_frac20", pa.float64(), nullable=True),
        pa.field("radius_frac50", pa.float64(), nullable=True),
        pa.field("radius_frac90", pa.float64(), nullable=True),
        pa.field("flux_petro", pa.float64(), nullable=True),
        pa.field("e_flux_petro", pa.float64(), nullable=True),
        pa.field("mag_petro", pa.float64(), nullable=True),
        pa.field("e_mag_petro", pa.float64(), nullable=True),
        pa.field("flux_kron", pa.float64(), nullable=True),
        pa.field("e_flux_kron", pa.float64(), nullable=True),
        pa.field("mag_kron", pa.float64(), nullable=True),
        pa.field("e_mag_kron", pa.float64(), nullable=True),
        pa.field("flux_ap02", pa.float64(), nullable=True),
        pa.field("e_flux_ap02", pa.float64(), nullable=True),
        pa.field("mag_apc02", pa.float64(), nullable=True),
        pa.field("e_mag_apc02", pa.float64(), nullable=True),
        pa.field("flux_ap03", pa.float64(), nullable=True),
        pa.field("e_flux_ap03", pa.float64(), nullable=True),
        pa.field("mag_apc03", pa.float64(), nullable=True),
        pa.field("e_mag_apc03", pa.float64(), nullable=True),
        pa.field("flux_ap04", pa.float64(), nullable=True),
        pa.field("e_flux_ap04", pa.float64(), nullable=True),
        pa.field("mag_apc04", pa.float64(), nullable=True),
        pa.field("e_mag_apc04", pa.float64(), nullable=True),
        pa.field("flux_ap05", pa.float64(), nullable=True),
        pa.field("e_flux_ap05", pa.float64(), nullable=True),
        pa.field("mag_apc05", pa.float64(), nullable=True),
        pa.field("e_mag_apc05", pa.float64(), nullable=True),
        pa.field("flux_ap06", pa.float64(), nullable=True),
        pa.field("e_flux_ap06", pa.float64(), nullable=True),
        pa.field("mag_apc06", pa.float64(), nullable=True),
        pa.field("e_mag_apc06", pa.float64(), nullable=True),
        pa.field("flux_ap08", pa.float64(), nullable=True),
        pa.field("e_flux_ap08", pa.float64(), nullable=True),
        pa.field("mag_apc08", pa.float64(), nullable=True),
        pa.field("e_mag_apc08", pa.float64(), nullable=True),
        pa.field("flux_ap10", pa.float64(), nullable=True),
        pa.field("e_flux_ap10", pa.float64(), nullable=True),
        pa.field("mag_apc10", pa.float64(), nullable=True),
        pa.field("e_mag_apc10", pa.float64(), nullable=True),
        pa.field("flux_ap15", pa.float64(), nullable=True),
        pa.field("e_flux_ap15", pa.float64(), nullable=True),
        pa.field("mag_apr15", pa.float64(), nullable=True),
        pa.field("e_mag_apr15", pa.float64(), nullable=True),
        pa.field("flux_ap20", pa.float64(), nullable=True),
        pa.field("e_flux_ap20", pa.float64(), nullable=True),
        pa.field("mag_apr20", pa.float64(), nullable=True),
        pa.field("e_mag_apr20", pa.float64(), nullable=True),
        pa.field("flux_ap30", pa.float64(), nullable=True),
        pa.field("e_flux_ap30", pa.float64(), nullable=True),
        pa.field("mag_apr30", pa.float64(), nullable=True),
        pa.field("e_mag_apr30", pa.float64(), nullable=True),
        pa.field("imaflags", pa.int64(), nullable=True),
        pa.field("x_mosaic", pa.float64(), nullable=True),
        pa.field("y_mosaic", pa.float64(), nullable=True),
        pa.field("chi2_psf", pa.float64(), nullable=True),
        pa.field("flux_psf", pa.float64(), nullable=True),
        pa.field("e_flux_psf", pa.float64(), nullable=True),
        pa.field("mag_psf", pa.float64(), nullable=True),
        pa.field("e_mag_psf", pa.float64(), nullable=True),
        pa.field("filter", pa.string()),
        pa.field("img_qual", pa.int32(), nullable=True),
        pa.field("use_in_clipped", pa.int32(), nullable=True),
        pa.field("object_id_local", pa.int64(), nullable=True),
    ]
)
pq.write_metadata(
    schema_from_pyarrow,
    "/epyc/data3/hats/raw/skymapper/photo_schema_from_pyarrow.parquet",
)

In [ ]:
# !wget https://skymapper.anu.edu.au/_data/DR4/SkyMapper.DR4.images.csv.gz

In [ ]:
images = pd.read_csv("SkyMapper.DR4.images.csv.gz")
# restrict to just the columns i want - hoping we get under 5M!

skinny = images[["image_id", "date", "fwhm_mean", "airmass"]]
skinny = skinny.set_index("image_id")
skinny.to_parquet("images.skinny2.parquet")

## Object Catalog

In [ ]:
schema_file = os.path.join(
    "/epyc/data3/hats/raw/skymapper/", "schema_from_pyarrow.parquet"
)

In [ ]:
with Client(
    local_directory="/astro/store/shiren/hats/skymapper/",
    n_workers=30,
    threads_per_worker=1,
) as client:
    args = ImportArguments(
        output_artifact_name="sky_mapper_dr4",
        input_path="/epyc/data3/hats/raw/skymapper/dr4_master",
        output_path="/astro/store/shire/hats/catalogs/v09/",
        tmp_dir="/astro/store/shiren/hats/skymapper/",
        use_schema_file=schema_file,
        file_reader=CsvReader(
            chunksize=250_000,
            compression="gzip",
            schema_file=schema_file,
        ),
        ra_column="raj2000",
        dec_column="dej2000",
        expected_total_rows=724_329_344,
        pixel_threshold=5_000_000,
        highest_healpix_order=8,
        drop_empty_siblings=True,
        skymap_alt_orders=[2, 4, 6],
        row_group_kwargs={"num_rows": 500_000},
        completion_email_address="delucchi@andrew.cmu.edu",
        progress_bar=True,
        simple_progress_bar=True,
    )
    pipeline_with_client(args, client)

In [ ]:
args = (
    CollectionArguments(
        completion_email_address="delucchi@andrew.cmu.edu",
        output_artifact_name="skymapper",
        output_path="/astro/store/shire/hats/catalogs/v09/",
        progress_bar=True,
        simple_progress_bar=True,
    )
    .catalog(output_artifact_name="sky_mapper_dr4")
    .add_margin(margin_threshold=5.0, is_default=True)
)

In [ ]:
with Client(
    local_directory="/astro/store/shiren/hats/skymapper/",
    n_workers=30,
    threads_per_worker=1,
) as client:
    pipeline_with_client(args, client)

In [ ]:
args = VerificationArguments(
    input_catalog_path="/astro/store/shire/hats/catalogs/v09/sky_mapper_dr4",
    output_path="./verification/skymapper_dr4_sep17",
)
pipeline(args)

## Photometry

In [ ]:
schema_file = os.path.join(
    "/astro/store/shiren/hats/skymapper/", "photo_schema_from_pyarrow.parquet"
)
schema_from_pyarrow = pq.read_metadata(schema_file).schema.to_arrow_schema()

In [ ]:
class SkymapperPhotometryCsvReader(InputReader):

    def __init__(
        self, column_names=None, convert_options=None, read_options=None, **kwargs
    ):
        self.kwargs = kwargs
        self.column_names = column_names
        self.convert_options = convert_options or csv.ConvertOptions()
        self.read_options = read_options or csv.ReadOptions(block_size=1048576 * 10)

    def read(self, input_file, read_columns=None):
        self.regular_file_exists(input_file, **self.kwargs)

        read_columns = read_columns or self.column_names
        convert_options = self.convert_options
        if read_columns:
            convert_options.include_columns = read_columns
            with csv.open_csv(
                input_file,
                convert_options=convert_options,
                read_options=self.read_options,
                **self.kwargs
            ) as reader:
                for next_chunk in reader:
                    if next_chunk is None:
                        break
                    table = pa.Table.from_batches([next_chunk])
                    table = table.replace_schema_metadata()
                    yield table
        else:
            lookup = pd.read_parquet(
                "/astro/store/shiren/hats/skymapper/images.skinny2.parquet"
            )
            with csv.open_csv(
                input_file,
                convert_options=convert_options,
                read_options=self.read_options,
                **self.kwargs
            ) as reader:
                for next_chunk in reader:
                    if next_chunk is None:
                        break

                    table = pa.Table.from_batches([next_chunk])
                    table = table.replace_schema_metadata()

                    chunk_lookup = lookup.loc[table.column("image_id")]
                    table = table.add_column(
                        2,
                        "date",
                        pa.array(list(chunk_lookup["date"].values), type=pa.float64()),
                    )
                    table = table.add_column(
                        3,
                        "fwhm_mean",
                        pa.array(
                            list(chunk_lookup["fwhm_mean"].values), type=pa.float64()
                        ),
                    )
                    table = table.add_column(
                        4,
                        "airmass",
                        pa.array(
                            list(chunk_lookup["airmass"].values), type=pa.float64()
                        ),
                    )
                    yield table

In [ ]:
args = ImportArguments(
    output_artifact_name="photometry",
    input_file_list=list(
        Path("/astro/store/shire/hats/delucchi_raw/skymapper").glob(
            "SkyMapper.DR4.pho*.csv.gz"
        )
    ),
    output_path="/astro/store/shire/hats/catalogs/v09/skymapper_take_5/",
    tmp_dir="/astro/store/shiren/hats/skymapper/",
    resume_tmp="/astro/store/shiren/hats/skymapper/",
    use_schema_file=schema_file,
    file_reader=SkymapperPhotometryCsvReader(
        convert_options=csv.ConvertOptions(column_types=schema_from_pyarrow)
    ),
    ra_column="ra_img",
    dec_column="decl_img",
    expected_total_rows=15_101_099_413,
    pixel_threshold=4_000_000,
    highest_healpix_order=10,
    drop_empty_siblings=True,
    skymap_alt_orders=[2, 4, 6, 8],
    row_group_kwargs={"num_rows": 500_000},
    completion_email_address="delucchi@andrew.cmu.edu",
    progress_bar=True,
    simple_progress_bar=True,
)

In [ ]:
with Client(
    local_directory="/astro/store/shiren/hats/skymapper/",
    n_workers=5,
    threads_per_worker=1,
    memory_limit="40GB",
) as client:
    pipeline_with_client(args, client)

## Nesting the lightcurves

1. Make a margin of the photometry
2. Join to the object catalog with an LSDB pipeline
3. Re-import, to make the most of the partitioning and compression
4. Create margin and collection
5. Run a verification
6. (not shown) move around the skymap files so the default uses order 6.

In [ ]:
args = MarginCacheArguments(
    input_catalog_path="/astro/store/shire/hats/catalogs/v09/skymapper_take_5/photometry",
    output_path="/astro/store/shire/hats/catalogs/v09/skymapper_take_5/",
    margin_threshold=15,
    output_artifact_name="photometry_15arcs",
    # resume=False,
    tmp_dir="/astro/store/shiren/hats/skymapper/",
    # resume_tmp="/astro/store/shiren/hats/skymapper/",
    completion_email_address="delucchi@andrew.cmu.edu",
    progress_bar=True,
    simple_progress_bar=True,
)

In [ ]:
with Client(
    local_directory="/astro/store/shiren/hats/skymapper/",
    n_workers=20,
    threads_per_worker=1,
) as client:
    pipeline_with_client(args, client)

In [ ]:
client = Client(
    local_directory="/astro/store/shiren/hats/skymapper/",
    n_workers=10,
    threads_per_worker=1,
)

In [ ]:
# Load object catalog
obj_cat = lsdb.read_hats("/astro/store/shire/hats/catalogs/v09/sky_mapper_dr4")

# Load and join each source catalog
src_cat = lsdb.read_hats(
    "/astro/store/shire/hats/catalogs/v09/skymapper_take_5/photometry",
    margin_cache="/astro/store/shire/hats/catalogs/v09/skymapper_take_5/photometry_15arcs",
)

In [ ]:
# Using map_partitions to drop rows with NaN in lc.mag
src_cat = src_cat.map_partitions(lambda df: df.dropna(subset=["object_id"]))


def _sort_nested_sources(df, source_cols: list[str], sort_col: str):
    for col in source_cols:
        flat = df[col].nest.to_flat()
        df = df.drop(columns=[col])
        df = df.join_nested(flat.sort_values([flat.index.name, sort_col]), col)
    return df


nested_cat = obj_cat.join_nested(
    src_cat,
    left_on="object_id",
    right_on="object_id",
    nested_column_name="photometry",
)

# Sort sources within each object by MJD
# source_cols = nested_columns
nested_cat = nested_cat.map_partitions(
    lambda df: _sort_nested_sources(df, ["photometry"], "date")
)

# Save intermediate and reimport with production settings
nested_cat.to_hats(
    "/astro/store/shire/hats/catalogs/v09/skymapper_take_5/",
    catalog_name="photometry_nested",
    resume=True,
)

In [ ]:
client.close()

In [ ]:
args = ImportArguments.reimport_from_hats(
    "/astro/store/shire/hats/catalogs/v09/skymapper_take_5/photometry_nested",
    output_dir="/astro/store/shire/hats/catalogs/v09/skymapper_take_5/",
    output_artifact_name="skymapper_dr4_lightcurves",
    tmp_dir="/astro/store/shiren/hats/skymapper/",
    resume_tmp="/astro/store/shiren/hats/skymapper/",
    expected_total_rows=724_329_344,
    pixel_threshold=800_000,
    byte_pixel_threshold=1024 * 1024 * 1024,
    highest_healpix_order=12,
    drop_empty_siblings=True,
    resume=False,
    skymap_alt_orders=[2, 4, 6],
    # row_group_kwargs={"num_rows": 200_000},
    completion_email_address="delucchi@andrew.cmu.edu",
    progress_bar=True,
    simple_progress_bar=True,
)

In [ ]:
with Client(
    local_directory="/astro/store/shiren/hats/skymapper/",
    n_workers=50,
    threads_per_worker=1,
    # memory_limit="80GB",
) as client:
    pipeline_with_client(args, client)

In [ ]:
args = (
    CollectionArguments(
        output_artifact_name="skymapper_dr4",
        output_path="/astro/store/shire/hats/catalogs/v09/skymapper_take_5",
        tmp_dir="/astro/store/shiren/hats/skymapper/",
        completion_email_address="delucchi@andrew.cmu.edu",
        progress_bar=True,
        simple_progress_bar=True,
    )
    .catalog(
        output_artifact_name="skymapper_dr4_lightcurves",
    )
    .add_margin(margin_threshold=5.0, is_default=True)
)

In [ ]:
with Client(
    local_directory="/astro/store/shiren/hats/skymapper/",
    n_workers=30,
    threads_per_worker=1,
    # memory_limit="80GB",
) as client:
    pipeline_with_client(args, client)